# AQ26 32C — GitHub Repository LAQN v3.5 + NASA Earthdata CMR Runbook

This notebook is a Colab-friendly runbook for the repository patch. It assumes you have merged the patch into the root of `AirQuality26_v2`.

In [ ]:
# Optional: mount Google Drive in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab or Drive mount skipped:', exc)


In [ ]:
from pathlib import Path
import os, json, subprocess, sys

# CHANGE THIS if your repo is elsewhere
REPO_ROOT = Path('/content/drive/MyDrive/AirQuality26_v2')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
print('REPO_ROOT =', REPO_ROOT)
print('scripts exist:', (REPO_ROOT / 'scripts').exists())


In [ ]:
# Local syntax/preflight checks
for rel in ['scripts/aq26_provider_laqn.py', 'scripts/aq26_provider_earthdata.py']:
    p = REPO_ROOT / rel
    print('\nChecking', p)
    assert p.exists(), f'Missing {rel}'
    subprocess.run([sys.executable, '-m', 'py_compile', str(p)], check=True)
print('Python compile checks passed.')


In [ ]:
# Run LAQN metadata probe locally. This uses the public ERG API and writes outputs/31_laqn.
cmd = [
    sys.executable, str(REPO_ROOT / 'scripts/aq26_provider_laqn.py'),
    '--repo-root', str(REPO_ROOT),
    '--config', 'configs/aq26_laqn.yml',
    '--group-name', 'London'
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
summary_path = REPO_ROOT / 'outputs/31_laqn/laqn_summary.json'
print(summary_path.read_text()[:2000])


In [ ]:
# Optional tiny LAQN historical probe after metadata is confirmed.
RUN_DATA_PROBE = False
if RUN_DATA_PROBE:
    cmd = [
        sys.executable, str(REPO_ROOT / 'scripts/aq26_provider_laqn.py'),
        '--repo-root', str(REPO_ROOT),
        '--config', 'configs/aq26_laqn.yml',
        '--group-name', 'London',
        '--run-data-probe',
        '--start-date', '2024-07-22',
        '--end-date', '2024-07-23'
    ]
    subprocess.run(cmd, check=True)


In [ ]:
# NASA Earthdata CMR discovery probe: catalogue only, no large downloads.
cmd = [
    sys.executable, str(REPO_ROOT / 'scripts/aq26_provider_earthdata.py'),
    '--repo-root', str(REPO_ROOT),
    '--config', 'configs/aq26_earthdata.yml',
    '--max-collections', '10',
    '--max-granules-per-collection', '3'
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
summary_path = REPO_ROOT / 'outputs/32_earthdata/earthdata_summary.json'
print(summary_path.read_text()[:2000])


## GitHub Actions run order

1. Run `AQ26 LAQN Provider Probe V3.5` with `run_data_probe=false`.
2. Run the same workflow with `run_data_probe=true` only after metadata is green.
3. Run `AQ26 NASA Earthdata CMR Probe`.
4. Commit outputs only when the metadata/readiness summary is sensible.